# Patient-adaptive model: tune modulators, then compare feature recipes

This notebook first selects patient modulators using **inner grouped FRDA validation only**, then evaluates the resulting adaptive model on held-out FRDA participants. It repeats the experiment for FRDA-only and control-aware feature recipes.

Controls have no disease duration or GAA repeat length. Control specificity is therefore a clearly labelled reference-profile analysis: observed control age is used when selected, while disease-only modulators are fixed to their outer-training FRDA means.


## 1. Data, folds, and guardrails


In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "src").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("Could not find the repository root")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data.trackfa_pairs import trackfa_pairs_to_long
from src.eval.control_aware_selection import wide_cohort_to_pair_long
from src.features.panels import a_priori_70_feature_names
from src.reporting.experiment_artifacts import (
    read_experiment_contract,
    read_table_artifact,
    write_table_artifact,
)

RUN_ID = "trackfa_70_feature_comparison_v1"
RUN_DIR = REPO_ROOT / "results" / "experiments" / RUN_ID
SELECTION_DIR = RUN_DIR / "selections"
manifest, folds = read_experiment_contract(RUN_DIR)
feature_names = a_priori_70_feature_names()
frda_recipe = read_table_artifact(
    SELECTION_DIR / "frda_only_features_by_fold.csv",
    schema="feature_recipe",
    manifest=manifest,
    panel=feature_names,
)
control_recipe = read_table_artifact(
    SELECTION_DIR / "control_aware_features_by_fold.csv",
    schema="feature_recipe",
    manifest=manifest,
    panel=feature_names,
)
feature_recipes = pd.concat([frda_recipe, control_recipe], ignore_index=True)

pairs = pd.read_csv(manifest["data_path"])
frda_long = trackfa_pairs_to_long(pairs)
wide_path = Path(manifest["data_path"]).with_name("trackfa_merged_wide.csv")
wide = pd.read_csv(wide_path, low_memory=False)
control_long = wide_cohort_to_pair_long(wide, feature_names, cohort_value=1)
control_meta = wide.loc[pd.to_numeric(wide["study_group"], errors="coerce").eq(1), ["ID", "age"]].copy()
control_meta["subject"] = control_meta["ID"].astype(str).str.replace(r"^TRACKFA_", "", regex=True)
control_meta["age"] = pd.to_numeric(control_meta["age"], errors="coerce")
control_long = control_long.merge(control_meta[["subject", "age"]].drop_duplicates("subject"), on="subject", how="left")

print(f"Run: {RUN_ID}")
display(pd.DataFrame([
    {"Cohort": "FRDA", "Participants": frda_long["subject"].nunique(), "Annual pairs": frda_long["pair_id"].nunique()},
    {"Cohort": "Control", "Participants": control_long["subject"].nunique(), "Annual pairs": control_long["pair_id"].nunique()},
]))
display(pd.DataFrame([
    {"Selection recipe": "FRDA-only", "Features per fold": int(frda_recipe.groupby("outer_fold")["selected"].sum().mode().iloc[0])},
    {"Selection recipe": "Control-aware", "Features per fold": int(control_recipe.groupby("outer_fold")["selected"].sum().mode().iloc[0])},
]))

from src.config import Config
from src.eval.recipe_models import DEFAULT_MODULATOR_SETS, run_adaptive_recipe_comparison

GUARDRAILS = pd.DataFrame([
    {"Step": "Feature recipe", "Chosen from": "Outer-training cohort data", "Never uses": "Outer-test data"},
    {"Step": "Modulator choice", "Chosen from": "Inner grouped FRDA CV", "Never uses": "Outer FRDA/control effects"},
    {"Step": "Imaging/modulator scaling", "Chosen from": "Outer-training FRDA", "Never uses": "Control values"},
    {"Step": "Control specificity", "Chosen from": "Frozen FRDA model", "Never uses": "Control model fitting"},
])
display(GUARDRAILS)


Run: trackfa_70_feature_comparison_v1


,Cohort,Participants,Annual pairs
0,FRDA,117,207
1,Control,95,190


,Selection recipe,Features per fold
0,FRDA-only,16
1,Control-aware,16


,Step,Chosen from,Never uses
0,Feature recipe,Outer-training cohort data,Outer-test data
1,Modulator choice,Inner grouped FRDA CV,Outer FRDA/control effects
2,Imaging/modulator scaling,Outer-training FRDA,Control values
3,Control specificity,Frozen FRDA model,Control model fitting


## 2. Inner-CV modulator screening

Candidate modulators are age, disease duration, GAA1, and their prespecified combinations. GAA2 is excluded from the primary screen because it changes the denominator; it can be added later as a denominator-matched sensitivity analysis.


In [2]:
adaptive_config = Config(
    random_state=int(manifest["seed"]),
    interaction_en_alpha_grid=(0.03, 0.1, 0.3, 1.0, 3.0),
    interaction_en_l1_ratio_grid=(0.0,),
    interaction_inner_cv_splits=3,
    interaction_z_clip=2.75,
    interaction_tune_inner_cv=True,
)
result = run_adaptive_recipe_comparison(
    frda_long,
    control_long,
    folds,
    feature_recipes,
    run_id=RUN_ID,
    modulator_sets=DEFAULT_MODULATOR_SETS,
    inner_folds=3,
    seed=int(manifest["seed"]),
    n_boot=500,
    config=adaptive_config,
)

MODEL_DIR = RUN_DIR / "models" / "patient_adaptive"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
write_table_artifact(MODEL_DIR / "oof_visit_scores.csv", result["oof_visit_scores"], schema="oof_visit_scores", manifest=manifest)
write_table_artifact(MODEL_DIR / "performance.csv", result["performance"], schema="performance", manifest=manifest)
write_table_artifact(MODEL_DIR / "coefficients.csv", result["coefficients"], schema="coefficients", manifest=manifest)
result["modulator_inner_scores"].to_csv(MODEL_DIR / "modulator_inner_scores.csv", index=False)
result["modulator_choices"].to_csv(MODEL_DIR / "modulator_choices.csv", index=False)
result["modulator_frequency"].to_csv(MODEL_DIR / "modulator_frequency.csv", index=False)
result["fold_parameters"].to_csv(MODEL_DIR / "fold_parameters.csv", index=False)
result["site_diagnostics"].to_csv(MODEL_DIR / "site_diagnostics.csv", index=False)
result["comparison"].to_csv(MODEL_DIR / "headline_comparison.csv", index=False)

screen = result["modulator_inner_scores"].rename(columns={
    "selection_strategy": "Feature selection", "outer_fold": "Fold", "modulators": "Modulator(s)",
    "mean_validation_annual_dz": "Inner annual d_z", "dz_v1_v2": "Inner V1-V2 d_z",
    "dz_v2_v3": "Inner V2-V3 d_z", "annual_interval_gap": "Inner gap",
    "n_participants": "Training participants", "n_pairs": "Training pairs",
})
print("Inner-validation modulator evidence")
display(screen[[
    "Feature selection", "Fold", "Modulator(s)", "Inner annual d_z", "Inner V1-V2 d_z",
    "Inner V2-V3 d_z", "Inner gap", "Training participants", "Training pairs",
]].round(3))
print("How often each modulator recipe was selected")
display(result["modulator_frequency"].round(3))


Inner-validation modulator evidence


,Feature selection,Fold,Modulator(s),Inner annual d_z,Inner V1-V2 d_z,Inner V2-V3 d_z,Inner gap,Training participants,Training pairs
0,frda_only,1,age,0.638,0.791,0.484,0.307,93,165
1,frda_only,1,disease_duration,0.695,0.855,0.535,0.320,93,165
2,frda_only,1,gaa_1,0.665,0.857,0.474,0.383,93,165
3,frda_only,1,"age,disease_duration",0.630,0.734,0.525,0.209,93,165
4,frda_only,1,"age,gaa_1",0.686,0.883,0.490,0.392,93,165
...,...,...,...,...,...,...,...,...,...
65,control_aware,5,gaa_1,0.759,0.834,0.684,0.150,94,167
66,control_aware,5,"age,disease_duration",0.650,0.584,0.715,0.131,94,167
67,control_aware,5,"age,gaa_1",0.639,0.586,0.693,0.107,94,167
68,control_aware,5,"disease_duration,gaa_1",0.620,0.609,0.632,0.022,94,167


How often each modulator recipe was selected


,comparison_mode,selection_strategy,modulators,folds_selected,selection_frequency
0,common_modulator,control_aware,disease_duration,2,0.4
1,common_modulator,control_aware,gaa_1,3,0.6
2,common_modulator,frda_only,disease_duration,2,0.4
3,common_modulator,frda_only,gaa_1,3,0.6
4,strategy_specific,control_aware,disease_duration,2,0.4
5,strategy_specific,control_aware,"disease_duration,gaa_1",1,0.2
6,strategy_specific,control_aware,gaa_1,2,0.4
7,strategy_specific,frda_only,disease_duration,1,0.2
8,strategy_specific,frda_only,"disease_duration,gaa_1",2,0.4
9,strategy_specific,frda_only,gaa_1,2,0.4


## 3. Held-out adaptive comparison

`strategy_specific` allows each feature recipe to choose its own best modulator inside every outer-training fold. `common_modulator` chooses the same modulator for both feature recipes from their combined inner evidence, isolating the feature-selection difference.


In [3]:
from src.eval.recipe_models import comparison_table

srm_performance = read_table_artifact(
    RUN_DIR / "models" / "srm_global_linear" / "performance.csv",
    schema="performance",
    manifest=manifest,
)
srm_baseline = comparison_table(srm_performance).copy()
srm_baseline["model"] = "global_srm_no_modulator"
combined_comparison = pd.concat([srm_baseline, result["comparison"]], ignore_index=True)

headline_columns = [
    "model", "selection_strategy",
    "frda_pooled_annual_d_z", "frda_pooled_annual_ci_low", "frda_pooled_annual_ci_high",
    "frda_pooled_annual_n_participants", "frda_pooled_annual_n_pairs",
    "control_pooled_annual_d_z", "control_pooled_annual_ci_low", "control_pooled_annual_ci_high",
    "control_pooled_annual_n_participants", "control_pooled_annual_n_pairs",
    "signed_frda_control_contrast", "absolute_control_d_z",
    "frda_v1_v2_d_z", "frda_v2_v3_d_z", "control_v1_v2_d_z", "control_v2_v3_d_z",
    "frda_interval_gap",
]
headline = combined_comparison[[c for c in headline_columns if c in combined_comparison]].rename(columns={
    "model": "Adaptive comparison", "selection_strategy": "Feature selection",
})
headline["Adaptive comparison"] = headline["Adaptive comparison"].map({
    "global_srm_no_modulator": "No modulator: matched global SRM",
    "patient_adaptive_strategy_specific": "Adaptive: strategy-specific modulator",
    "patient_adaptive_common_modulator": "Adaptive: common modulator",
}).fillna(headline["Adaptive comparison"])
print("Zero-modulator baseline and patient-adaptive held-out comparison")
display(headline.round(3))
headline.to_csv(MODEL_DIR / "adaptive_vs_no_modulator_comparison.csv", index=False)


Zero-modulator baseline and patient-adaptive held-out comparison


,Adaptive comparison,Feature selection,frda_pooled_annual_d_z,frda_pooled_annual_ci_low,frda_pooled_annual_ci_high,frda_pooled_annual_n_participants,frda_pooled_annual_n_pairs,control_pooled_annual_d_z,control_pooled_annual_ci_low,control_pooled_annual_ci_high,control_pooled_annual_n_participants,control_pooled_annual_n_pairs,signed_frda_control_contrast,absolute_control_d_z,frda_v1_v2_d_z,frda_v2_v3_d_z,control_v1_v2_d_z,control_v2_v3_d_z,frda_interval_gap
0,No modulator: matched global SRM,control_aware,0.749,0.614,0.900,117,207,-0.011,-0.168,0.168,67,126,0.760,0.011,0.944,0.579,-0.019,-0.003,0.365
1,No modulator: matched global SRM,frda_only,0.749,0.610,0.912,117,207,0.033,-0.121,0.222,67,126,0.715,0.033,0.959,0.569,0.007,0.059,0.390
2,Adaptive: common modulator,control_aware,0.662,0.502,0.832,117,207,0.076,-0.098,0.262,67,126,0.586,0.076,0.802,0.532,0.124,0.032,0.270
3,Adaptive: common modulator,frda_only,0.709,0.570,0.882,117,207,0.114,-0.057,0.285,67,126,0.595,0.114,0.798,0.625,0.163,0.068,0.172
4,Adaptive: strategy-specific modulator,control_aware,0.662,0.502,0.830,117,207,0.076,-0.099,0.265,67,126,0.586,0.076,0.793,0.537,0.136,0.021,0.256
5,Adaptive: strategy-specific modulator,frda_only,0.654,0.504,0.826,117,207,0.076,-0.086,0.244,67,126,0.579,0.076,0.728,0.577,0.085,0.066,0.150


## 4. Control reference profile and site diagnostic

The control result is not a patient-specific adaptive score when GAA1 or disease duration is selected. It asks whether the FRDA-trained adaptive imaging direction, evaluated at an average FRDA disease profile, also changes in healthy controls.


In [4]:
policy = result["fold_parameters"][[
    "model", "selection_strategy", "outer_fold", "modulators", "control_modulator_policy",
    "train_frda_participants", "train_frda_pairs", "alpha", "l1_ratio",
]].rename(columns={
    "model": "Adaptive comparison", "selection_strategy": "Feature selection", "outer_fold": "Fold",
    "modulators": "Modulator(s)", "control_modulator_policy": "Control scoring policy",
    "train_frda_participants": "Train FRDA N", "train_frda_pairs": "Train FRDA pairs",
})
display(policy)
site_display = result["site_diagnostics"].rename(columns={
    "model": "Adaptive comparison", "selection_strategy": "Feature selection", "cohort": "Cohort",
    "site_r2_delta": "Site partial R2", "site_p_value": "Site p-value",
})
display(site_display[[c for c in ["Adaptive comparison", "Feature selection", "Cohort", "n", "site_levels", "Site partial R2", "Site p-value"] if c in site_display]].round(3))


,Adaptive comparison,Feature selection,Fold,Modulator(s),Control scoring policy,Train FRDA N,Train FRDA pairs,alpha,l1_ratio
0,patient_adaptive_strategy_specific,frda_only,1,"disease_duration,gaa_1",disease_duration=FRDA-training mean; gaa_1=FRD...,93,165,0.30,0.0
1,patient_adaptive_strategy_specific,frda_only,2,gaa_1,gaa_1=FRDA-training mean,93,163,0.30,0.0
2,patient_adaptive_strategy_specific,frda_only,3,"disease_duration,gaa_1",disease_duration=FRDA-training mean; gaa_1=FRD...,94,167,0.30,0.0
3,patient_adaptive_strategy_specific,frda_only,4,gaa_1,gaa_1=FRDA-training mean,94,166,3.00,0.0
4,patient_adaptive_strategy_specific,frda_only,5,disease_duration,disease_duration=FRDA-training mean,94,167,1.00,0.0
5,patient_adaptive_strategy_specific,control_aware,1,disease_duration,disease_duration=FRDA-training mean,93,165,0.03,0.0
6,patient_adaptive_strategy_specific,control_aware,2,disease_duration,disease_duration=FRDA-training mean,93,163,0.30,0.0
7,patient_adaptive_strategy_specific,control_aware,3,gaa_1,gaa_1=FRDA-training mean,94,167,0.30,0.0
8,patient_adaptive_strategy_specific,control_aware,4,"disease_duration,gaa_1",disease_duration=FRDA-training mean; gaa_1=FRD...,94,166,1.00,0.0
9,patient_adaptive_strategy_specific,control_aware,5,gaa_1,gaa_1=FRDA-training mean,94,167,1.00,0.0


,Adaptive comparison,Feature selection,Cohort,n,site_levels,Site partial R2,Site p-value
0,patient_adaptive_strategy_specific,frda_only,FRDA,207,6,0.070,0.012
1,patient_adaptive_strategy_specific,frda_only,Control,126,6,0.030,0.586
2,patient_adaptive_strategy_specific,control_aware,FRDA,207,6,0.066,0.017
3,patient_adaptive_strategy_specific,control_aware,Control,126,6,0.020,0.791
4,patient_adaptive_common_modulator,frda_only,FRDA,207,6,0.068,0.014
5,patient_adaptive_common_modulator,frda_only,Control,126,6,0.033,0.542
6,patient_adaptive_common_modulator,control_aware,FRDA,207,6,0.070,0.012
7,patient_adaptive_common_modulator,control_aware,Control,126,6,0.020,0.791


## 5. Coefficient interpretation and deployment boundary

The exported adaptive coefficient is the effective standardised imaging weight at the outer-training FRDA mean modulator profile. It is conditional on the other MRI features and must not be read as a marginal feature-change direction.

A later locked-winner interpretation notebook may refit the chosen feature and modulator configuration on all eligible FRDA data for deployment. New-patient scoring then requires the frozen feature order, MRI means/SDs, modulator means/SDs, clipping rule, main coefficients, and interaction coefficients. That full-data model must not replace the held-out results above.


In [5]:
coef_preview = result["coefficients"].copy()
coef_preview["absolute_coefficient"] = coef_preview["coefficient"].abs()
coef_preview = coef_preview.sort_values("absolute_coefficient", ascending=False, kind="mergesort")
display(coef_preview[[
    "model", "selection_strategy", "outer_fold", "feature", "coefficient",
    "modulator_reference_profile",
]].head(30).round(3))
print("Machine-readable artifacts:", MODEL_DIR)


,model,selection_strategy,outer_fold,feature,coefficient,modulator_reference_profile
87,patient_adaptive_strategy_specific,control_aware,1,Lateral_Ventricle,2.792,disease_duration=11.3315
247,patient_adaptive_common_modulator,control_aware,1,Lateral_Ventricle,2.792,disease_duration=11.3315
224,patient_adaptive_common_modulator,frda_only,5,Cerebellum_Cortex_CerebNet,-2.554,gaa_1=600.299
228,patient_adaptive_common_modulator,frda_only,5,Lateral_Ventricle,1.961,gaa_1=600.299
16,patient_adaptive_strategy_specific,frda_only,2,Cerebellum_Cortex_CerebNet,-1.956,gaa_1=595.503
32,patient_adaptive_strategy_specific,frda_only,3,Cerebellum_Cortex_CerebNet,-1.871,disease_duration=11.0407; gaa_1=593.198
192,patient_adaptive_common_modulator,frda_only,3,Cerebellum_Cortex_CerebNet,-1.865,gaa_1=593.198
0,patient_adaptive_strategy_specific,frda_only,1,Cerebellum_Cortex_CerebNet,-1.797,disease_duration=11.3315; gaa_1=594.036
128,patient_adaptive_strategy_specific,control_aware,4,Cerebellum_Cortex_CerebNet,-1.727,disease_duration=11.4205; gaa_1=624.042
288,patient_adaptive_common_modulator,control_aware,4,Cerebellum_Cortex_CerebNet,-1.705,gaa_1=624.042


Machine-readable artifacts: /Users/robertwang/Documents/New_project/biomarkers/results/experiments/trackfa_70_feature_comparison_v1/models/patient_adaptive
